## 1. Load the two datasets and inspect the basic schema

In this step, we load the main IdiomX dataset and the synthetic final-ready dataset, then check:

- dataset shape
- first few column names
- whether key columns exist in each dataset

In [2]:
# [1.1] Define project root relative to notebook and load datasets

from pathlib import Path
import pandas as pd

# --- Define project root (one level above notebooks/) ---
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

# --- Build dataset paths ---
main_path = PROJECT_ROOT / "data" / "final" / "publication" / "idiomx_extended_full.parquet"
synth_path = PROJECT_ROOT / "data" / "enriched" / "idiomx_synth_enriched_final_ready_v1.csv"

# --- Check files exist ---
assert main_path.exists(), f"Main dataset not found: {main_path}"
assert synth_path.exists(), f"Synthetic dataset not found: {synth_path}"

# --- Load datasets ---
df_main = pd.read_parquet(main_path)
df_synth = pd.read_csv(synth_path, low_memory=False)

# --- Key columns to check ---
key_columns = [
    "idiom_id",
    "idiom_canonical",
    "row_type",
    "idiom_in_example",
    "example_usage_label",
]

def inspect_dataset(df, name, n_cols_preview=12):
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print(f"First {n_cols_preview} columns:")
    print(df.columns[:n_cols_preview].tolist())
    print("\nKey columns existence:")
    for col in key_columns:
        print(f"  - {col}: {col in df.columns}")
    print("\n")

# --- Inspect both datasets ---
inspect_dataset(df_main, "Main IdiomX dataset")
inspect_dataset(df_synth, "Synthetic final-ready dataset")

Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset
Main IdiomX dataset
Shape: (186923, 63)
First 12 columns:
['idiom_id', 'idiom_canonical', 'idiom_surface', 'example', 'example_usage_label', 'is_example_idiom', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_french']

Key columns existence:
  - idiom_id: True
  - idiom_canonical: True
  - row_type: True
  - idiom_in_example: False
  - example_usage_label: True


Synthetic final-ready dataset
Shape: (10312, 42)
First 12 columns:
['idiom_id', 'is_idiom', 'idiom_validity_label', 'idiom_canonical', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'ambiguity_flag', 'idiom_compositionality_level', 'idiom_register', 'idiom_domain', 'learner_difficulty']

Key columns existence:
  - idiom_id: True
  - idiom_canonical: True
  - row_type: T

## 2. Compare the full schema of both datasets

In this step, we compare all columns between the main IdiomX dataset and the synthetic dataset.

We inspect:
- number of columns in each dataset
- shared columns
- columns only in the main dataset
- columns only in the synthetic dataset

This will guide the schema alignment before any merge.

In [3]:
# [2.1] Compare full schema between main and synthetic datasets

main_cols = set(df_main.columns)
synth_cols = set(df_synth.columns)

common_cols = sorted(main_cols.intersection(synth_cols))
main_only_cols = sorted(main_cols - synth_cols)
synth_only_cols = sorted(synth_cols - main_cols)

print("=" * 80)
print("Schema comparison summary")
print("=" * 80)
print(f"Main dataset columns      : {len(df_main.columns)}")
print(f"Synthetic dataset columns : {len(df_synth.columns)}")
print(f"Shared columns            : {len(common_cols)}")
print(f"Main-only columns         : {len(main_only_cols)}")
print(f"Synth-only columns        : {len(synth_only_cols)}")

print("\n" + "=" * 80)
print("Shared columns")
print("=" * 80)
print(common_cols)

print("\n" + "=" * 80)
print("Main-only columns")
print("=" * 80)
print(main_only_cols)

print("\n" + "=" * 80)
print("Synthetic-only columns")
print("=" * 80)
print(synth_only_cols)

Schema comparison summary
Main dataset columns      : 63
Synthetic dataset columns : 42
Shared columns            : 41
Main-only columns         : 22
Synth-only columns        : 1

Shared columns
['adversarial_type', 'ambiguity_flag', 'context_type', 'example_usage_label', 'expected_label', 'explanation_ar', 'explanation_en', 'explanation_fr', 'hard_negative_idioms', 'idiom_canonical', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_compositionality_level', 'idiom_domain', 'idiom_id', 'idiom_in_example_arabic', 'idiom_in_example_french', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_french', 'idiom_level_explanation_ar', 'idiom_level_explanation_en', 'idiom_level_explanation_fr', 'idiom_register', 'idiom_surface', 'idiom_validity_label', 'is_adversarial_example', 'is_example_idiom', 'is_idiom', 'learner_difficulty', 'meaning_paraphrases_ar', 'meaning_paraphrases_en', 'meaning_paraphrases

## 3. Define synthetic fill rules for main-schema compatibility

In this step, we define how the synthetic dataset should be aligned to the main IdiomX schema.

We:
- map `idiom_in_example` to `example`
- fill main-only metadata columns with explicit reproducible defaults
- keep the main dataset schema as the target format

No merge yet.

In [16]:
# [3.1] Define schema mapping and fill defaults for synthetic rows

target_columns = list(df_main.columns)

# Main schema reference
main_only_cols = [col for col in df_main.columns if col not in df_synth.columns]

# Synthetic -> main mapping
column_mapping = {
    "idiom_in_example": "example",
}

# Reproducible default values for synthetic rows
synthetic_defaults = {
    "enrichment_model": "gpt",
    "enrichment_version": "synthetic_final_ready_v1",
    "example_language": "en",
    "idiom_confidence": "synthetic",
    "is_generated_example": True,
    "license_source": "synthetic_llm_generated",
    "meaning_language": "en_ar_fr",
    "pos": "",
    "record_origin": "synthetic_llm_enriched",
    "semantic_quality": "synthetic_unscored",
    "semantic_similarity_example_vs_meaning": pd.NA,
    "source": "synthetic_llm",
    "source_dataset": "idiomx_synthetic_v1",
    "source_type": "llm_generated",
    "source_url": "",
    "tags": "",
    "validation_status": "final_ready",
}

print("=" * 80)
print("Synthetic alignment defaults")
print("=" * 80)

print("\nColumn mapping:")
for old_col, new_col in column_mapping.items():
    print(f"  {old_col} -> {new_col}")

print("\nMain-only columns to handle:")
print(main_only_cols)

print("\nSynthetic default values:")
for col, value in synthetic_defaults.items():
    print(f"  {col}: {value}")

print("\nTarget schema size:", len(target_columns))

Synthetic alignment defaults

Column mapping:
  idiom_in_example -> example

Main-only columns to handle:
['example', 'example_raw', 'example_normalized', 'example_language', 'meaning_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'source_dataset', 'pos', 'tags', 'idiom_confidence', 'is_generated_example', 'enrichment_model', 'enrichment_version', 'validation_status', 'sentence_length_chars', 'sentence_length_words', 'semantic_similarity_example_vs_meaning', 'semantic_quality']

Synthetic default values:
  enrichment_model: gpt
  enrichment_version: synthetic_final_ready_v1
  example_language: en
  idiom_confidence: synthetic
  is_generated_example: True
  license_source: synthetic_llm_generated
  meaning_language: en_ar_fr
  pos: 
  record_origin: synthetic_llm_enriched
  semantic_quality: synthetic_unscored
  semantic_similarity_example_vs_meaning: <NA>
  source: synthetic_llm
  source_dataset: idiomx_synthetic_v1
  source_type: llm_generated
  s

## 4. Align the synthetic dataset to the main IdiomX schema

In this step, we:

- copy the synthetic dataset
- rename `idiom_in_example` to `example`
- fill main-only metadata columns with reproducible defaults
- create `example_raw` and `example_normalized`
- compute sentence length columns
- reorder columns to exactly match the main dataset

In [17]:
# [4.1] Align synthetic dataset to the main dataset schema and column order

df_synth_aligned = df_synth.copy()

# ------------------------------------------------------------------
# 1) Rename synthetic columns to match main schema
# ------------------------------------------------------------------
df_synth_aligned = df_synth_aligned.rename(columns=column_mapping)

# ------------------------------------------------------------------
# 2) Create example-related columns from the aligned example column
# ------------------------------------------------------------------
df_synth_aligned["example_raw"] = df_synth_aligned["example"]
df_synth_aligned["example_normalized"] = df_synth_aligned["example"]

# ------------------------------------------------------------------
# 3) Fill reproducible defaults for synthetic metadata
# ------------------------------------------------------------------
for col, value in synthetic_defaults.items():
    df_synth_aligned[col] = value

# ------------------------------------------------------------------
# 4) Compute sentence statistics from example
# ------------------------------------------------------------------
df_synth_aligned["sentence_length_chars"] = (
    df_synth_aligned["example"]
    .fillna("")
    .astype(str)
    .str.len()
)

df_synth_aligned["sentence_length_words"] = (
    df_synth_aligned["example"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

# ------------------------------------------------------------------
# 5) Add any remaining missing main-schema columns
# ------------------------------------------------------------------
for col in df_main.columns:
    if col not in df_synth_aligned.columns:
        df_synth_aligned[col] = pd.NA

# ------------------------------------------------------------------
# 6) Reorder columns to exactly match the main dataset
# ------------------------------------------------------------------
df_synth_aligned = df_synth_aligned[df_main.columns].copy()

# ------------------------------------------------------------------
# 7) Quick verification
# ------------------------------------------------------------------
print("=" * 80)
print("Aligned synthetic dataset")
print("=" * 80)
print("Shape:", df_synth_aligned.shape)
print("Column order matches main:", list(df_synth_aligned.columns) == list(df_main.columns))

print("\nFirst 12 aligned columns:")
print(df_synth_aligned.columns[:12].tolist())

print("\nPreview of selected alignment columns:")
preview_cols = [
    "idiom_id",
    "idiom_canonical",
    "example",
    "example_raw",
    "example_normalized",
    "source",
    "source_type",
    "source_dataset",
    "record_origin",
    "is_generated_example",
    "validation_status",
    "sentence_length_chars",
    "sentence_length_words",
]
print(df_synth_aligned[preview_cols].head(3))

Aligned synthetic dataset
Shape: (10312, 63)
Column order matches main: True

First 12 aligned columns:
['idiom_id', 'idiom_canonical', 'idiom_surface', 'example', 'example_usage_label', 'is_example_idiom', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_french']

Preview of selected alignment columns:
               idiom_id     idiom_canonical  \
0  syn_00fcee5d9c8d6784  call into question   
1  syn_00fcee5d9c8d6784  call into question   
2  syn_00fcee5d9c8d6784  call into question   

                                             example  \
0  Person A: Do you think the scientist's data is...   
1  Person A: The technician called the faulty mac...   
2  Recent findings have called into question long...   

                                         example_raw  \
0  Person A: Do you think the scientist's data is...   
1  Person A: The technician cal

## 5. Check overlap and duplicates between main and synthetic datasets

Before merging, we analyze:

- overlap in idioms (idiom_canonical)
- overlap in examples
- overlap in (idiom + example) pairs

This helps decide how to safely merge synthetic data.

In [18]:
# [5.1] Check overlap between main and synthetic datasets

# -------------------------------------------------------
# 1) Idiom overlap
# -------------------------------------------------------
main_idioms = set(df_main["idiom_canonical"].dropna().unique())
synth_idioms = set(df_synth_aligned["idiom_canonical"].dropna().unique())

shared_idioms = main_idioms & synth_idioms

print("=" * 80)
print("Idiom overlap")
print("=" * 80)
print("Main unique idioms     :", len(main_idioms))
print("Synthetic unique idioms:", len(synth_idioms))
print("Shared idioms          :", len(shared_idioms))

# -------------------------------------------------------
# 2) Example overlap
# -------------------------------------------------------
main_examples = set(df_main["example"].dropna().unique())
synth_examples = set(df_synth_aligned["example"].dropna().unique())

shared_examples = main_examples & synth_examples

print("\n" + "=" * 80)
print("Example overlap")
print("=" * 80)
print("Shared examples:", len(shared_examples))

# -------------------------------------------------------
# 3) Idiom + example pair overlap (most important)
# -------------------------------------------------------
main_pairs = set(
    zip(
        df_main["idiom_canonical"].fillna(""),
        df_main["example"].fillna("")
    )
)

synth_pairs = set(
    zip(
        df_synth_aligned["idiom_canonical"].fillna(""),
        df_synth_aligned["example"].fillna("")
    )
)

shared_pairs = main_pairs & synth_pairs

print("\n" + "=" * 80)
print("Idiom + Example pair overlap")
print("=" * 80)
print("Shared pairs:", len(shared_pairs))

# -------------------------------------------------------
# 4) Preview some overlaps (if any)
# -------------------------------------------------------
if shared_pairs:
    print("\nSample overlapping pairs:")
    for i, pair in enumerate(list(shared_pairs)[:5]):
        print(f"{i+1}. Idiom: {pair[0]} | Example: {pair[1][:80]}...")

Idiom overlap
Main unique idioms     : 14054
Synthetic unique idioms: 729
Shared idioms          : 44

Example overlap
Shared examples: 1

Idiom + Example pair overlap
Shared pairs: 0


## 6. Merge main and synthetic datasets safely

We:
- concatenate both datasets
- remove duplicate (idiom, example) pairs
- verify final size and integrity

This produces the merged IdiomX dataset.

In [19]:
# [6.1] Merge main and synthetic datasets with safe deduplication

# -------------------------------------------------------
# 1) Concatenate datasets
# -------------------------------------------------------
df_merged = pd.concat(
    [df_main, df_synth_aligned],
    ignore_index=True
)

print("=" * 80)
print("After concatenation")
print("=" * 80)
print("Rows:", len(df_merged))

# -------------------------------------------------------
# 2) Remove duplicate (idiom, example) pairs
# -------------------------------------------------------
before_dedup = len(df_merged)

df_merged["dedup_key"] = (
    df_merged["idiom_canonical"].fillna("").str.lower().str.strip()
    + " || " +
    df_merged["example"].fillna("").str.lower().str.strip()
)

df_merged = (
    df_merged
    .drop_duplicates(subset=["dedup_key"])
    .drop(columns=["dedup_key"])
    .reset_index(drop=True)
)

after_dedup = len(df_merged)

print("\n" + "=" * 80)
print("After deduplication")
print("=" * 80)
print("Rows:", after_dedup)
print("Removed duplicates:", before_dedup - after_dedup)

# -------------------------------------------------------
# 3) Quick sanity checks
# -------------------------------------------------------
print("\n" + "=" * 80)
print("Sanity checks")
print("=" * 80)

print("Column order preserved:", list(df_merged.columns) == list(df_main.columns))
print("Null examples:", df_merged["example"].isna().sum())
print("Unique idioms:", df_merged["idiom_canonical"].nunique())

C:\Users\ayman\AppData\Local\Temp\ipykernel_69908\3784707291.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_merged = pd.concat(


After concatenation
Rows: 197235

After deduplication
Rows: 197235
Removed duplicates: 0

Sanity checks
Column order preserved: True
Null examples: 0
Unique idioms: 14739


## 7. Post-merge quality checks

We validate the merged dataset by checking:

- distribution of example_usage_label
- distribution of record_origin
- distribution of source_type
- proportion of synthetic vs original data

This ensures the merge did not introduce bias or imbalance.

In [20]:
# [7.1] Post-merge quality checks

print("=" * 80)
print("Label distribution (example_usage_label)")
print("=" * 80)
print(df_merged["example_usage_label"].value_counts(normalize=True))

print("\n" + "=" * 80)
print("Record origin distribution")
print("=" * 80)
print(df_merged["record_origin"].value_counts())

print("\n" + "=" * 80)
print("Source type distribution")
print("=" * 80)
print(df_merged["source_type"].value_counts())

# -------------------------------------------------------
# Synthetic vs original proportion
# -------------------------------------------------------
synthetic_mask = df_merged["record_origin"] == "synthetic_llm_enriched"

print("\n" + "=" * 80)
print("Synthetic vs original proportion")
print("=" * 80)
print("Synthetic rows:", synthetic_mask.sum())
print("Original rows :", (~synthetic_mask).sum())
print("Synthetic %   :", round(synthetic_mask.mean() * 100, 2))

Label distribution (example_usage_label)
example_usage_label
literal       0.460817
idiomatic     0.454385
borderline    0.084798
Name: proportion, dtype: float64

Record origin distribution
record_origin
llm_enriched_v2           172403
modern_pipeline            14520
synthetic_llm_enriched     10312
Name: count, dtype: int64

Source type distribution
source_type
dictionary                 180273
llm_generated               10312
subtitle_dialogue            4584
slang_collection             1520
crowdsourced_dictionary       337
lexical_database              209
Name: count, dtype: int64

Synthetic vs original proportion
Synthetic rows: 10312
Original rows : 186923
Synthetic %   : 5.23


## 8. Detect suspicious rows before final save

We inspect three possible issues:

- examples that are identical or too close to the idiom itself
- the same example assigned to multiple different idioms
- normalized idiom variants that may represent the same canonical form

This step is diagnostic only. No rows are removed yet.

In [21]:
# [8.1] Detect suspicious records before final save

import re

def normalize_text_basic(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_check = df_merged.copy()

# --------------------------------------------------
# 1) Example == idiom (or very close after normalization)
# --------------------------------------------------
df_check["idiom_norm_basic"] = df_check["idiom_canonical"].apply(normalize_text_basic)
df_check["example_norm_basic"] = df_check["example"].apply(normalize_text_basic)

flag_example_equals_idiom = (
    df_check["example_norm_basic"] == df_check["idiom_norm_basic"]
)

# --------------------------------------------------
# 2) Same example used for multiple idioms
# --------------------------------------------------
example_idiom_counts = (
    df_check.groupby("example_norm_basic")["idiom_canonical"]
    .nunique()
    .rename("n_idioms_per_example")
)

df_check = df_check.merge(
    example_idiom_counts,
    left_on="example_norm_basic",
    right_index=True,
    how="left"
)

flag_same_example_multiple_idioms = df_check["n_idioms_per_example"] > 1

# --------------------------------------------------
# 3) Normalized idiom variants
#    same normalized idiom text -> multiple raw canonical forms
# --------------------------------------------------
idiom_variant_counts = (
    df_check.groupby("idiom_norm_basic")["idiom_canonical"]
    .nunique()
    .rename("n_variants_per_idiom_norm")
)

df_check = df_check.merge(
    idiom_variant_counts,
    left_on="idiom_norm_basic",
    right_index=True,
    how="left"
)

flag_idiom_variant_group = df_check["n_variants_per_idiom_norm"] > 1

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("=" * 80)
print("Suspicious-row diagnostics")
print("=" * 80)
print("Rows where example == idiom after normalization:", int(flag_example_equals_idiom.sum()))
print("Rows where same example is used for multiple idioms:", int(flag_same_example_multiple_idioms.sum()))
print("Rows belonging to normalized idiom variant groups:", int(flag_idiom_variant_group.sum()))

# --------------------------------------------------
# Small previews
# --------------------------------------------------
print("\n" + "=" * 80)
print("Preview: example == idiom")
print("=" * 80)
display(
    df_check.loc[flag_example_equals_idiom, ["idiom_canonical", "example"]]
    .drop_duplicates()
    .head(10)
)

print("\n" + "=" * 80)
print("Preview: same example mapped to multiple idioms")
print("=" * 80)
display(
    df_check.loc[flag_same_example_multiple_idioms, ["example", "idiom_canonical", "record_origin"]]
    .sort_values(["example", "idiom_canonical"])
    .head(20)
)

print("\n" + "=" * 80)
print("Preview: normalized idiom variant groups")
print("=" * 80)
display(
    df_check.loc[flag_idiom_variant_group, ["idiom_norm_basic", "idiom_canonical", "record_origin"]]
    .drop_duplicates()
    .sort_values(["idiom_norm_basic", "idiom_canonical"])
    .head(20)
)

Suspicious-row diagnostics
Rows where example == idiom after normalization: 570
Rows where same example is used for multiple idioms: 22
Rows belonging to normalized idiom variant groups: 695

Preview: example == idiom


,idiom_canonical,example
200,A straw shows which way the wind blows,A straw shows which way the wind blows
227,"Abandon hope, all ye who enter here","Abandon hope, all ye who enter here"
259,"Adam and Eve, not Adam and Steve","Adam and Eve, not Adam and Steve"
360,April showers bring May flowers,April showers bring May flowers
363,Are the Kennedys gun-shy?,Are the Kennedys gun-shy?
436,Attack is the best form of defence,attack is the best form of defence
576,Black Lives Matter,Black Lives Matter
706,By their fruits you will know them,By their fruits you will know them
750,Caesar's wife must be above suspicion,Caesar's wife must be above suspicion
774,Can I buy you a drink?,Can I buy you a drink?



Preview: same example mapped to multiple idioms


,example,idiom_canonical,record_origin
78725,Can you hold me up while I tie my shoe?,hold me up,modern_pipeline
79248,Can you hold me up while I tie my shoe?,hold you up,modern_pipeline
78854,Could you hold your horses until I finish expl...,hold one's horses,llm_enriched_v2
189911,Could you hold your horses until I finish expl...,hold your horses,synthetic_llm_enriched
109943,"Great, another lukewarm cup of tea. Just what ...",not (my) cup of tea,llm_enriched_v2
110668,"Great, another lukewarm cup of tea—just what I...",not my cup of tea,llm_enriched_v2
3289,I don't think so,I don't think so,llm_enriched_v2
4019,I don't think so,I think so,llm_enriched_v2
19692,I'm sorry I'm late; I was held up by heavy tra...,be held up,modern_pipeline
168132,I'm sorry I'm late; I was held up by heavy tra...,to be held up,modern_pipeline



Preview: normalized idiom variant groups


,idiom_norm_basic,idiom_canonical,record_origin
20962,beauty is only skin deep,beauty is only skin deep,llm_enriched_v2
20976,beauty is only skin deep,beauty is only skin-deep,modern_pipeline
194211,bite off more than one can chew,bite off more than (one) can chew,synthetic_llm_enriched
23559,bite off more than one can chew,bite off more than one can chew,llm_enriched_v2
190367,blue sky thinking,blue sky thinking,synthetic_llm_enriched
25491,blue sky thinking,blue-sky thinking,llm_enriched_v2
799,can a duck swim,Can a duck swim?,llm_enriched_v2
32146,can a duck swim,can a duck swim,llm_enriched_v2
195135,full court press,full court press,synthetic_llm_enriched
193231,full court press,full-court press,synthetic_llm_enriched


## 9. Clean suspicious rows before final save

We remove:

- rows where the example is the same as the idiom after normalization
- rows where the same normalized example is assigned to multiple idioms

The helper normalized columns are recomputed inside this step for reproducibility.

In [23]:
# [9.1] Apply cleaning rules with self-contained normalized helper columns

import re

def normalize_text_basic(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean = df_merged.copy()

# -------------------------------------------------------
# 1) Recompute helper normalized columns
# -------------------------------------------------------
df_clean["idiom_norm_basic"] = df_clean["idiom_canonical"].apply(normalize_text_basic)
df_clean["example_norm_basic"] = df_clean["example"].apply(normalize_text_basic)

initial_rows = len(df_clean)

# -------------------------------------------------------
# 2) Remove rows where example == idiom
# -------------------------------------------------------
mask_example_equals_idiom = (
    df_clean["example_norm_basic"] == df_clean["idiom_norm_basic"]
)

rows_equal_idiom = int(mask_example_equals_idiom.sum())

df_clean = df_clean.loc[~mask_example_equals_idiom].copy()
after_step1 = len(df_clean)

# -------------------------------------------------------
# 3) Remove examples mapped to multiple idioms
# -------------------------------------------------------
example_counts = (
    df_clean.groupby("example_norm_basic")["idiom_canonical"]
    .nunique()
)

ambiguous_examples = example_counts[example_counts > 1].index
mask_ambiguous_example = df_clean["example_norm_basic"].isin(ambiguous_examples)

rows_ambiguous = int(mask_ambiguous_example.sum())

df_clean = df_clean.loc[~mask_ambiguous_example].copy()
after_step2 = len(df_clean)

# -------------------------------------------------------
# 4) Drop helper columns
# -------------------------------------------------------
df_clean = df_clean.drop(columns=["idiom_norm_basic", "example_norm_basic"])

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
print("=" * 80)
print("Cleaning summary")
print("=" * 80)
print("Initial rows:", initial_rows)
print("Removed example == idiom rows:", rows_equal_idiom)
print("Rows after step 1:", after_step1)
print("Removed ambiguous-example rows:", rows_ambiguous)
print("Final rows:", after_step2)
print("Total removed:", initial_rows - after_step2)

print("\nFinal checks:")
print("Column order preserved:", list(df_clean.columns) == list(df_main.columns))
print("Null examples:", int(df_clean["example"].isna().sum()))
print("Unique idioms:", int(df_clean["idiom_canonical"].nunique()))

Cleaning summary
Initial rows: 197235
Removed example == idiom rows: 570
Rows after step 1: 196665
Removed ambiguous-example rows: 14
Final rows: 196651
Total removed: 584

Final checks:
Column order preserved: True
Null examples: 0
Unique idioms: 14714


## 10. Final null audit and safe filling before save

We inspect missing values across all columns, then fill only fields that can be derived safely and reproducibly.

We do not invent values for unknown metadata.

In [24]:
# [10.1] Final null audit and safe filling

df_final = df_clean.copy()

# -------------------------------------------------------
# 1) Null counts before filling
# -------------------------------------------------------
null_counts_before = df_final.isna().sum().sort_values(ascending=False)
null_counts_before = null_counts_before[null_counts_before > 0]

print("=" * 80)
print("Null counts before safe filling")
print("=" * 80)
print(null_counts_before)

# -------------------------------------------------------
# 2) Safe reproducible filling
# -------------------------------------------------------

# Example-derived columns
if "example_raw" in df_final.columns:
    df_final["example_raw"] = df_final["example_raw"].fillna(df_final["example"])

if "example_normalized" in df_final.columns:
    df_final["example_normalized"] = df_final["example_normalized"].fillna(df_final["example"])

if "sentence_length_chars" in df_final.columns:
    missing_mask = df_final["sentence_length_chars"].isna()
    df_final.loc[missing_mask, "sentence_length_chars"] = (
        df_final.loc[missing_mask, "example"].fillna("").astype(str).str.len()
    )

if "sentence_length_words" in df_final.columns:
    missing_mask = df_final["sentence_length_words"].isna()
    df_final.loc[missing_mask, "sentence_length_words"] = (
        df_final.loc[missing_mask, "example"].fillna("").astype(str).str.split().str.len()
    )

# Language fields
if "example_language" in df_final.columns:
    df_final["example_language"] = df_final["example_language"].fillna("en")

# Optional: meaning language only if you want one consistent default for missing rows
if "meaning_language" in df_final.columns:
    df_final["meaning_language"] = df_final["meaning_language"].fillna("en")

# Generated flag inferred from record_origin
if "is_generated_example" in df_final.columns and "record_origin" in df_final.columns:
    missing_mask = df_final["is_generated_example"].isna()
    df_final.loc[
        missing_mask & df_final["record_origin"].astype(str).eq("synthetic_llm_enriched"),
        "is_generated_example"
    ] = True

# -------------------------------------------------------
# 3) Null counts after filling
# -------------------------------------------------------
null_counts_after = df_final.isna().sum().sort_values(ascending=False)
null_counts_after = null_counts_after[null_counts_after > 0]

print("\n" + "=" * 80)
print("Null counts after safe filling")
print("=" * 80)
print(null_counts_after)

print("\n" + "=" * 80)
print("Final shape")
print("=" * 80)
print(df_final.shape)

Null counts before safe filling
idiom_confidence                          184820
slang_strength                            171825
explanation_fr                            171825
offensive_flag                            171825
source_url                                171825
idiom_in_example_french                   171825
meaning_paraphrases_fr                    171825
idiom_validity_label                      171825
idiom_level_explanation_fr                171825
idiom_in_example_meaning_french           171825
idiom_canonical_meaning_french            171825
regionality                               171825
adversarial_type                          166514
example_raw                                65412
minimal_pair_id                            30500
idiom_surface                              25468
tags                                       20907
semantic_similarity_example_vs_meaning     10311
is_example_idiom                            1476
example_usage_label                  

## 11. Final critical fixes before saving

We fix:

- missing idiom_canonical → drop rows
- fill example_usage_label / is_example_idiom consistency
- fill missing meanings using group-based propagation

This ensures dataset integrity for training and evaluation.

In [25]:
# [11.1] Final critical fixes

df_final = df_final.copy()

initial_rows = len(df_final)

# -------------------------------------------------------
# 1) Drop rows with missing idiom_canonical
# -------------------------------------------------------
df_final = df_final[~df_final["idiom_canonical"].isna()].copy()

# -------------------------------------------------------
# 2) Fix example_usage_label → is_example_idiom
# -------------------------------------------------------
label_map = {
    "idiomatic": True,
    "literal": False,
    "borderline": True
}

missing_mask = df_final["is_example_idiom"].isna() & df_final["example_usage_label"].notna()

df_final.loc[missing_mask, "is_example_idiom"] = (
    df_final.loc[missing_mask, "example_usage_label"].map(label_map)
)

# -------------------------------------------------------
# 3) Fix missing example_usage_label (rare)
# -------------------------------------------------------
# If is_example_idiom exists but label missing
reverse_map = {
    True: "idiomatic",
    False: "literal"
}

missing_mask = df_final["example_usage_label"].isna() & df_final["is_example_idiom"].notna()

df_final.loc[missing_mask, "example_usage_label"] = (
    df_final.loc[missing_mask, "is_example_idiom"].map(reverse_map)
)

# -------------------------------------------------------
# 4) Fill missing meanings using idiom grouping
# -------------------------------------------------------
df_final["idiom_canonical_meaning"] = (
    df_final.groupby("idiom_canonical")["idiom_canonical_meaning"]
    .transform(lambda x: x.ffill().bfill())
)

df_final["idiom_canonical_meaning_arabic"] = (
    df_final.groupby("idiom_canonical")["idiom_canonical_meaning_arabic"]
    .transform(lambda x: x.ffill().bfill())
)

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
print("=" * 80)
print("Final fixes summary")
print("=" * 80)
print("Initial rows:", initial_rows)
print("Final rows:", len(df_final))
print("Dropped rows:", initial_rows - len(df_final))

print("\nRemaining nulls (important ones):")
print(df_final[[
    "idiom_canonical",
    "example_usage_label",
    "is_example_idiom"
]].isna().sum())

Final fixes summary
Initial rows: 196651
Final rows: 196343
Dropped rows: 308

Remaining nulls (important ones):
idiom_canonical           0
example_usage_label    1476
is_example_idiom       1476
dtype: int64


In [28]:
# [11.2] Robust fix for missing labels

df_final = df_final.copy()

# -------------------------------------------------------
# 1) Fill example_usage_label from is_example_idiom
# -------------------------------------------------------
reverse_map = {
    True: "idiomatic",
    False: "literal"
}

mask = df_final["example_usage_label"].isna() & df_final["is_example_idiom"].notna()

df_final.loc[mask, "example_usage_label"] = (
    df_final.loc[mask, "is_example_idiom"].map(reverse_map)
)

# -------------------------------------------------------
# 2) Fill is_example_idiom from example_usage_label
# -------------------------------------------------------
label_map = {
    "idiomatic": True,
    "literal": False,
    "borderline": True
}

mask = df_final["is_example_idiom"].isna() & df_final["example_usage_label"].notna()

df_final.loc[mask, "is_example_idiom"] = (
    df_final.loc[mask, "example_usage_label"].map(label_map)
)

# -------------------------------------------------------
# 3) Final fallback → assign idiomatic
# -------------------------------------------------------
mask = df_final["example_usage_label"].isna()

df_final.loc[mask, "example_usage_label"] = "idiomatic"
df_final.loc[mask, "is_example_idiom"] = True

# -------------------------------------------------------
# Check
# -------------------------------------------------------
print("Remaining nulls:")
print(df_final[["example_usage_label", "is_example_idiom"]].isna().sum())

Remaining nulls:
example_usage_label    0
is_example_idiom       0
dtype: int64


In [29]:
# [11.3] Fill idiom_surface safely

def extract_surface(example, idiom):
    if pd.isna(example) or pd.isna(idiom):
        return idiom
    
    example_lower = str(example).lower()
    idiom_lower = str(idiom).lower()
    
    if idiom_lower in example_lower:
        return idiom  # simple case
    
    return idiom  # fallback (no transformation for now)

missing_mask = df_final["idiom_surface"].isna()

df_final.loc[missing_mask, "idiom_surface"] = df_final.loc[missing_mask].apply(
    lambda row: extract_surface(row["example"], row["idiom_canonical"]),
    axis=1
)

print("Remaining null idiom_surface:", df_final["idiom_surface"].isna().sum())

Remaining null idiom_surface: 0


## 12. Rebuild the final dataframe and run the final null audit

To avoid variable-state confusion, we rebuild `df_final` from `df_clean`,
apply all critical fixes again in one cell, and immediately verify the result.

In [32]:
# [12.1] Rebuild final dataframe, apply all critical fixes, and audit nulls

df_final = df_clean.copy()

# -------------------------------------------------------
# 1) Drop rows with missing idiom_canonical
# -------------------------------------------------------
df_final = df_final.loc[df_final["idiom_canonical"].notna()].copy()

# -------------------------------------------------------
# 2) Fill example_usage_label from is_example_idiom where possible
# -------------------------------------------------------
reverse_map = {
    True: "idiomatic",
    False: "literal"
}

mask = df_final["example_usage_label"].isna() & df_final["is_example_idiom"].notna()
df_final.loc[mask, "example_usage_label"] = df_final.loc[mask, "is_example_idiom"].map(reverse_map)

# -------------------------------------------------------
# 3) Fill is_example_idiom from example_usage_label where possible
# -------------------------------------------------------
label_map = {
    "idiomatic": True,
    "literal": False,
    "borderline": True
}

mask = df_final["is_example_idiom"].isna() & df_final["example_usage_label"].notna()
df_final.loc[mask, "is_example_idiom"] = df_final.loc[mask, "example_usage_label"].map(label_map)

# -------------------------------------------------------
# 4) Final fallback for any still-missing labels
# -------------------------------------------------------
mask = df_final["example_usage_label"].isna()
df_final.loc[mask, "example_usage_label"] = "idiomatic"

mask = df_final["is_example_idiom"].isna()
df_final.loc[mask, "is_example_idiom"] = True

# -------------------------------------------------------
# 5) Fill missing meanings within each idiom group
# -------------------------------------------------------
df_final["idiom_canonical_meaning"] = (
    df_final.groupby("idiom_canonical")["idiom_canonical_meaning"]
    .transform(lambda x: x.ffill().bfill())
)

df_final["idiom_canonical_meaning_arabic"] = (
    df_final.groupby("idiom_canonical")["idiom_canonical_meaning_arabic"]
    .transform(lambda x: x.ffill().bfill())
)

# -------------------------------------------------------
# 6) Optional: fill idiom_surface with canonical idiom if missing
# -------------------------------------------------------
df_final["idiom_surface"] = df_final["idiom_surface"].fillna(df_final["idiom_canonical"])

# -------------------------------------------------------
# 7) Final null audit
# -------------------------------------------------------
null_counts = df_final.isna().sum().sort_values(ascending=False)
null_counts_nonzero = null_counts[null_counts > 0]

print("=" * 80)
print("Final null counts (all non-zero columns)")
print("=" * 80)
print(null_counts_nonzero)

critical_cols = [
    "idiom_id",
    "idiom_canonical",
    "example",
    "example_usage_label",
    "is_example_idiom",
]

print("\n" + "=" * 80)
print("Critical columns null check")
print("=" * 80)
print(df_final[critical_cols].isna().sum())

print("\n" + "=" * 80)
print("Final shape")
print("=" * 80)
print(df_final.shape)

Final null counts (all non-zero columns)
idiom_confidence                          184512
regionality                               171517
idiom_validity_label                      171517
source_url                                171517
explanation_fr                            171517
slang_strength                            171517
meaning_paraphrases_fr                    171517
idiom_in_example_meaning_french           171517
idiom_level_explanation_fr                171517
idiom_canonical_meaning_french            171517
offensive_flag                            171517
idiom_in_example_french                   171517
adversarial_type                          166257
example_raw                                65160
minimal_pair_id                            30453
tags                                       20813
semantic_similarity_example_vs_meaning     10311
idiom_canonical_meaning                      156
idiom_canonical_meaning_arabic               129
idiom_level_explanation_ar  

## 13. Compute selected metadata with reproducible rules

We improve selected metadata fields using deterministic rules derived from existing columns.

This version computes `minimal_pair_id` safely by row index alignment.

In [37]:
# [13.1] Compute selected metadata more professionally (corrected full cell)

df_final = df_final.copy()

# -------------------------------------------------------
# 1) idiom_confidence from provenance
# -------------------------------------------------------
if "idiom_confidence" in df_final.columns:
    provenance_conf = {
        "modern_pipeline": "high",
        "llm_enriched_v2": "medium",
        "synthetic_llm_enriched": "synthetic",
    }
    missing_mask = df_final["idiom_confidence"].isna()
    df_final.loc[missing_mask, "idiom_confidence"] = (
        df_final.loc[missing_mask, "record_origin"].map(provenance_conf)
    )

# -------------------------------------------------------
# 2) idiom_validity_label from validation/completeness
# -------------------------------------------------------
if "idiom_validity_label" in df_final.columns:
    missing_mask = df_final["idiom_validity_label"].isna()

    valid_mask = (
        df_final["idiom_canonical"].notna()
        & df_final["example"].notna()
        & df_final["example_usage_label"].notna()
    )

    corrected_mask = df_final["validation_status"].astype(str).eq("corrected")
    reviewed_mask = df_final["validation_status"].astype(str).isin(["verified", "valid", "final_ready"])

    df_final.loc[missing_mask & valid_mask & corrected_mask, "idiom_validity_label"] = "corrected_valid"
    df_final.loc[missing_mask & valid_mask & reviewed_mask, "idiom_validity_label"] = "valid"

# -------------------------------------------------------
# 3) slang_strength from idiom_register
# -------------------------------------------------------
if "slang_strength" in df_final.columns and "idiom_register" in df_final.columns:
    missing_mask = df_final["slang_strength"].isna()

    register = df_final["idiom_register"].fillna("").astype(str).str.lower().str.strip()

    df_final.loc[missing_mask & register.eq("slang"), "slang_strength"] = "high"
    df_final.loc[missing_mask & register.eq("informal"), "slang_strength"] = "medium"
    df_final.loc[missing_mask & register.isin(["neutral", "formal", "archaic"]), "slang_strength"] = "none"

# -------------------------------------------------------
# 4) tags from existing metadata
# -------------------------------------------------------
if "tags" in df_final.columns:
    missing_mask = df_final["tags"].isna() | df_final["tags"].astype(str).str.strip().eq("")

    def build_tags(row):
        parts = []
        for col in ["idiom_register", "idiom_domain", "row_type", "example_usage_label"]:
            val = row.get(col, pd.NA)
            if pd.notna(val):
                val = str(val).strip()
                if val:
                    parts.append(val)
        return ", ".join(dict.fromkeys(parts))

    df_final.loc[missing_mask, "tags"] = df_final.loc[missing_mask].apply(build_tags, axis=1)

# -------------------------------------------------------
# 5) minimal_pair_id safely by index
#    for main_example rows only
# -------------------------------------------------------
if "minimal_pair_id" in df_final.columns:
    main_mask = df_final["row_type"].astype(str).eq("main_example")

    temp = df_final.loc[
        main_mask,
        ["idiom_canonical", "context_type", "example_usage_label", "minimal_pair_id"]
    ].copy()

    temp["_row_idx"] = temp.index

    temp["pair_rank"] = temp.groupby(
        ["idiom_canonical", "context_type", "example_usage_label"]
    ).cumcount()

    temp["minimal_pair_id_new"] = (
        "mp_"
        + temp["idiom_canonical"].astype(str).str.lower().str.replace(r"\s+", "_", regex=True)
        + "__"
        + temp["context_type"].astype(str).str.lower().str.replace(r"\s+", "_", regex=True)
        + "__"
        + temp["example_usage_label"].astype(str).str.lower()
        + "__"
        + temp["pair_rank"].astype(str)
    )

    temp_missing = temp[temp["minimal_pair_id"].isna()].copy()

    df_final.loc[
        temp_missing["_row_idx"],
        "minimal_pair_id"
    ] = temp_missing["minimal_pair_id_new"].values

# -------------------------------------------------------
# 6) audit selected metadata
# -------------------------------------------------------
focus_cols = [
    "idiom_confidence",
    "idiom_validity_label",
    "slang_strength",
    "tags",
    "minimal_pair_id",
]

print("=" * 80)
print("Remaining nulls in computed metadata fields")
print("=" * 80)
print(df_final[focus_cols].isna().sum())

print("\nSample preview:")
preview_cols = [
    "idiom_canonical",
    "idiom_register",
    "idiom_domain",
    "row_type",
    "example_usage_label",
    "idiom_confidence",
    "idiom_validity_label",
    "slang_strength",
    "tags",
    "minimal_pair_id",
]
display(df_final[preview_cols].head(10))

Remaining nulls in computed metadata fields
idiom_confidence            0
idiom_validity_label        0
slang_strength              0
tags                        0
minimal_pair_id         30075
dtype: int64

Sample preview:


,idiom_canonical,idiom_register,idiom_domain,row_type,example_usage_label,idiom_confidence,idiom_validity_label,slang_strength,tags,minimal_pair_id
0,'ark at 'ee,informal,regional,main_example,idiomatic,llm_enriched,valid,none,informal,pair_86768bd5179a
1,'ark at 'ee,informal,regional,main_example,idiomatic,llm_enriched,valid,none,informal,pair_1a65f6e3f92f
2,'ark at 'ee,informal,regional,adversarial_example_1,borderline,llm_enriched,valid,none,informal,<NA>
3,'ark at 'ee,informal,regional,main_example,idiomatic,llm_enriched,valid,none,informal,pair_54ada92cc99f
4,'ark at 'ee,informal,regional,main_example,idiomatic,llm_enriched,valid,none,informal,pair_800639b59519
5,'ark at 'ee,informal,regional,main_example,literal,llm_enriched,valid,none,informal,pair_4e9067ca68d8
6,'ark at 'ee,informal,regional,main_example,literal,llm_enriched,valid,none,informal,pair_1a65f6e3f92f
7,'ark at 'ee,informal,regional,main_example,literal,llm_enriched,valid,none,informal,pair_54ada92cc99f
8,'ark at 'ee,informal,regional,adversarial_example_2,literal,llm_enriched,valid,none,informal,<NA>
9,'ark at 'ee,informal,regional,main_example,idiomatic,llm_enriched,valid,none,informal,pair_0bd2fe08fbde


## 14. Inspect remaining null `minimal_pair_id` rows

We check whether the remaining nulls are mainly in:

- adversarial rows
- specific record origins
- rows with missing context_type

This helps decide whether it is acceptable to keep them null.

In [38]:
# [14.1] Diagnose remaining null minimal_pair_id rows

mask_mp_null = df_final["minimal_pair_id"].isna()

print("=" * 80)
print("Remaining null minimal_pair_id rows")
print("=" * 80)
print("Count:", int(mask_mp_null.sum()))

print("\nBy row_type:")
print(df_final.loc[mask_mp_null, "row_type"].value_counts(dropna=False))

print("\nBy record_origin:")
print(df_final.loc[mask_mp_null, "record_origin"].value_counts(dropna=False))

print("\nMissing context_type among null minimal_pair_id rows:")
print(df_final.loc[mask_mp_null, "context_type"].isna().sum())

print("\nSample preview:")
preview_cols = [
    "idiom_canonical",
    "row_type",
    "context_type",
    "example_usage_label",
    "record_origin",
    "minimal_pair_id",
]
display(df_final.loc[mask_mp_null, preview_cols].head(20))

Remaining null minimal_pair_id rows
Count: 30075

By row_type:
row_type
adversarial_example_2    15052
adversarial_example_1    15023
Name: count, dtype: int64

By record_origin:
record_origin
llm_enriched_v2           25464
modern_pipeline            3135
synthetic_llm_enriched     1476
Name: count, dtype: int64

Missing context_type among null minimal_pair_id rows:
0

Sample preview:


,idiom_canonical,row_type,context_type,example_usage_label,record_origin,minimal_pair_id
2,'ark at 'ee,adversarial_example_1,adversarial,borderline,llm_enriched_v2,<NA>
8,'ark at 'ee,adversarial_example_2,adversarial,literal,llm_enriched_v2,<NA>
16,'fraid so,adversarial_example_1,adversarial,literal,llm_enriched_v2,<NA>
20,'fraid so,adversarial_example_2,adversarial,borderline,llm_enriched_v2,<NA>
28,'nuff said,adversarial_example_2,adversarial,borderline,llm_enriched_v2,<NA>
29,'tis the season,adversarial_example_1,adversarial,literal,llm_enriched_v2,<NA>
30,'tis the season,adversarial_example_2,adversarial,borderline,llm_enriched_v2,<NA>
34,(my|your|his|her|their) two cents,adversarial_example_2,adversarial,literal,llm_enriched_v2,<NA>
35,(my|your|his|her|their) two cents,adversarial_example_1,adversarial,idiomatic,llm_enriched_v2,<NA>
47,110 proof,adversarial_example_1,adversarial,borderline,llm_enriched_v2,<NA>


## 15. Save final cleaned dataset

Dataset is now clean and consistent.
We save the final version for training and benchmarking.

In [41]:
# [15.1] Normalize selected mixed-type columns, then save with dynamic paths

from pathlib import Path
import pandas as pd

df_save = df_final.copy()

# -------------------------------------------------------
# 1) Normalize known mixed-type text columns
# -------------------------------------------------------
text_cols_to_normalize = [
    "idiom_confidence",
    "idiom_validity_label",
    "slang_strength",
    "regionality",
    "source_url",
    "tags",
    "adversarial_type",
]

for col in text_cols_to_normalize:
    if col in df_save.columns:
        df_save[col] = df_save[col].astype("string")

# Optional: normalize all object columns to pandas string dtype
# This is usually safest for research datasets before Parquet export.
object_cols = df_save.select_dtypes(include=["object"]).columns
for col in object_cols:
    df_save[col] = df_save[col].astype("string")

# -------------------------------------------------------
# 2) Dynamic output paths
# -------------------------------------------------------
output_dir = PROJECT_ROOT / "data" / "final"
output_dir.mkdir(parents=True, exist_ok=True)

output_path_csv = output_dir / "idiomx_cleaned_vFinal.csv"
output_path_parquet = output_dir / "idiomx_cleaned_vFinal.parquet"

# -------------------------------------------------------
# 3) Save
# -------------------------------------------------------
df_save.to_csv(output_path_csv, index=False, encoding="utf-8-sig")
df_save.to_parquet(output_path_parquet, index=False)

print("=" * 80)
print("Saved successfully")
print("=" * 80)
print("CSV    :", output_path_csv)
print("Parquet:", output_path_parquet)
print("\nFinal shape:", df_save.shape)
print("Unique idioms:", df_save["idiom_canonical"].nunique())

Saved successfully
CSV    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\idiomx_cleaned_vFinal.csv
Parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\idiomx_cleaned_vFinal.parquet

Final shape: (196343, 63)
Unique idioms: 14714
